<a href="https://www.kaggle.com/code/archanakashiboina/assignment16-2303a51329?scriptVersionId=346974490" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = [
    f for f in Path("/kaggle/input/datasets/paultimothymooney/blood-cells").rglob("*")
    if f.suffix.lower() in image_extensions
]

print("Total images found:", len(image_files))

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

dataset_path = Path("/kaggle/input/datasets/paultimothymooney/blood-cells")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Get all images
image_files = [
    f for f in dataset_path.rglob("*")
    if f.is_file() and f.suffix.lower() in image_extensions
]

print("Total images:", len(image_files))

# Show image paths to understand dataset structure
print("\nSample image paths:")
for img in image_files[:10]:
    print(img)

In [ ]:
data = []

for img in image_files:
    data.append({
        "filepath": str(img),
        "filename": img.name,
        "parent_folder": img.parent.name
    })

df = pd.DataFrame(data)

print(df.head())
print("\nTotal Images:", len(df))
print("\nFolders containing images:")
print(df["parent_folder"].value_counts())

In [ ]:
from PIL import Image
from tqdm import tqdm

valid_files = []
corrupted_files = []

for filepath in tqdm(df["filepath"]):
    try:
        with Image.open(filepath) as img:
            img.verify()
        valid_files.append(filepath)
    except Exception:
        corrupted_files.append(filepath)

print("Valid images:", len(valid_files))
print("Corrupted images:", len(corrupted_files))

if corrupted_files:
    print("\nSample corrupted files:")
    for f in corrupted_files[:10]:
        print(f)

In [ ]:
df_clean = df[df["filepath"].isin(valid_files)].copy()

print("Images after corrupted image cleaning:", len(df_clean))
print("\nClass distribution after cleaning:")
print(df_clean["parent_folder"].value_counts())

In [ ]:
import hashlib
from tqdm import tqdm

def get_file_hash(filepath):
    hash_md5 = hashlib.md5()

    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()

hashes = {}

for filepath in tqdm(df_clean["filepath"]):
    file_hash = get_file_hash(filepath)

    if file_hash not in hashes:
        hashes[file_hash] = filepath

print("Total cleaned images:", len(df_clean))
print("Unique images based on exact file hash:", len(hashes))
print("Exact duplicate images:", len(df_clean) - len(hashes))

In [ ]:
unique_files = list(hashes.values())

df_unique = df_clean[df_clean["filepath"].isin(unique_files)].copy()

print("Final unique images:", len(df_unique))

print("\nFinal class distribution:")
class_counts = df_unique["parent_folder"].value_counts()
print(class_counts)

print("\nNumber of classes:", df_unique["parent_folder"].nunique())

In [ ]:
import matplotlib.pyplot as plt

class_counts = df_unique["parent_folder"].value_counts()

plt.figure(figsize=(10, 5))

plt.bar(class_counts.index, class_counts.values)

plt.title("Blood Cell Dataset Class Distribution")
plt.xlabel("Blood Cell Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)

for i, value in enumerate(class_counts.values):
    plt.text(i, value + 50, str(value), ha="center")

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 70% train and 30% temporary
train_df, temp_df = train_test_split(
    df_unique,
    test_size=0.30,
    stratify=df_unique["parent_folder"],
    random_state=42
)

# Second split: 15% validation and 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["parent_folder"],
    random_state=42
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nPercentage split:")
total = len(df_unique)

print("Train:", round(len(train_df) / total * 100, 2), "%")
print("Validation:", round(len(val_df) / total * 100, 2), "%")
print("Test:", round(len(test_df) / total * 100, 2), "%")

In [ ]:
split_distribution = pd.DataFrame({
    "Train": train_df["parent_folder"].value_counts(),
    "Validation": val_df["parent_folder"].value_counts(),
    "Test": test_df["parent_folder"].value_counts()
}).fillna(0).astype(int)

print(split_distribution)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit using all class names
label_encoder.fit(df_unique["parent_folder"])

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label"] = label_encoder.transform(train_df["parent_folder"])
val_df["label"] = label_encoder.transform(val_df["parent_folder"])
test_df["label"] = label_encoder.transform(test_df["parent_folder"])

class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("Classes:")
for i, class_name in enumerate(class_names):
    print(f"{i} -> {class_name}")

print("\nNumber of classes:", num_classes)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)

    return image, label


def create_dataset(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("TensorFlow datasets created successfully!")

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
layers.Dense(8, activation="softmax")

In [ ]:
import time
import gc

EPOCHS = 3
LEARNING_RATE = 0.001

# Use sparse labels (0 to 7)
loss_function = tf.keras.losses.SparseCategoricalCrossentropy()

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1
    )
]

print("Training configuration ready")
print("Classes:", num_classes)
print("Epochs:", EPOCHS)
print("Image Size:", IMG_SIZE)
print("Batch Size:", BATCH_SIZE)

In [ ]:
results = {}
trained_models = {}

def train_and_evaluate(model_name, model):

    print("\n" + "=" * 60)
    print("Training:", model_name)
    print("=" * 60)

    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    training_time = time.time() - start_time

    # Evaluate on test set
    test_loss, test_accuracy = model.evaluate(
        test_ds,
        verbose=0
    )

    # Parameter count
    parameter_count = model.count_params()

    results[model_name] = {
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "parameters": parameter_count,
        "training_time_seconds": training_time,
        "history": history.history
    }

    trained_models[model_name] = model

    print("\nResults for", model_name)
    print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
    print("Parameters:", f"{parameter_count:,}")
    print("Training Time:", round(training_time / 60, 2), "minutes")

    return model

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

# Pretrained MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False


# Build model
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

mobilenet_model = tf.keras.Model(
    inputs,
    outputs,
    name="MobileNetV2"
)

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=loss_function,
    metrics=["accuracy"]
)

mobilenet_model.summary()

In [ ]:
mobilenet_model = train_and_evaluate(
    "MobileNetV2",
    mobilenet_model
)

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

# Load pretrained ResNet50
base_model = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers for faster training
base_model.trainable = False

# Build model
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# ResNet preprocessing
x = tf.keras.applications.resnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

resnet50_model = tf.keras.Model(
    inputs,
    outputs,
    name="ResNet50"
)

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=loss_function,
    metrics=["accuracy"]
)

resnet50_model.summary()

In [ ]:
print(train_ds)
print(val_ds)
print(test_ds)
print(train_and_evaluate)

In [ ]:
import os

print(os.path.exists("/content/blood_cell_dataset"))

In [ ]:
from pathlib import Path
import pandas as pd

dataset_path = Path("/content/blood_cell_dataset")
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = [
    f for f in dataset_path.rglob("*")
    if f.is_file() and f.suffix.lower() in image_extensions
]

data = []

for img in image_files:
    data.append({
        "filepath": str(img),
        "parent_folder": img.parent.name
    })

df_unique = pd.DataFrame(data)

print("Total images:", len(df_unique))
print(df_unique["parent_folder"].value_counts())

In [ ]:
import os
import glob

print("ZIP files:")
print(glob.glob("/content/**/*.zip", recursive=True))

print("\nFolders/files inside /content:")
print(os.listdir("/content"))

print("\nDataset folder structure:")
for root, dirs, files in os.walk("/content/blood_cell_dataset"):
    level = root.replace("/content/blood_cell_dataset", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    # Show only a few files
    for file in files[:3]:
        print(f"{indent}    {file}")

    # Prevent an extremely long output
    if level >= 4:
        dirs[:] = []

In [ ]:
import zipfile
import glob
import os

zip_files = glob.glob("/content/*.zip")

print("ZIP files found:", zip_files)

for file in zip_files:
    print("\nFile:", file)
    print("Size:", os.path.getsize(file) / (1024 * 1024), "MB")
    print("Valid ZIP:", zipfile.is_zipfile(file))

In [ ]:
import zipfile
import os
import shutil

zip_path = zip_files[-1]
extract_path = "/kaggle/input/datasets/paultimothymooney/blood-cells"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = [
    f for f in Path("/content/blood_cell_dataset").rglob("*")
    if f.is_file() and f.suffix.lower() in image_extensions
]

print("Total images:", len(image_files))

In [ ]:
import pandas as pd
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Dataset location
dataset_path = Path("/kaggle/input/datasets/paultimothymooney/blood-cells")

# Find all images
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = [
    f for f in dataset_path.rglob("*")
    if f.is_file() and f.suffix.lower() in image_extensions
]

# Create dataframe
df_unique = pd.DataFrame({
    "filepath": [str(f) for f in image_files],
    "parent_folder": [f.parent.name for f in image_files]
})

print("Total images:", len(df_unique))
print("\nClass distribution:")
print(df_unique["parent_folder"].value_counts())


# ============================================================
# 70% TRAIN / 15% VALIDATION / 15% TEST
# ============================================================

train_df, temp_df = train_test_split(
    df_unique,
    test_size=0.30,
    stratify=df_unique["parent_folder"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["parent_folder"],
    random_state=42
)

print("\nDataset Split:")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))


# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

label_encoder.fit(df_unique["parent_folder"])

train_df["label"] = label_encoder.transform(
    train_df["parent_folder"]
)

val_df["label"] = label_encoder.transform(
    val_df["parent_folder"]
)

test_df["label"] = label_encoder.transform(
    test_df["parent_folder"]
)

class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("\nClasses:", class_names)
print("Number of classes:", num_classes)


# ============================================================
# TENSORFLOW DATASETS
# ============================================================

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


def create_dataset(dataframe, shuffle=False):

    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            min(len(dataframe), 5000),
            seed=42
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("\nSUCCESS! TensorFlow datasets created.")
print(train_ds)

In [ ]:
from tensorflow.keras import layers
import time

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

EPOCHS = 3

results = {}
trained_models = {}

loss_function = tf.keras.losses.SparseCategoricalCrossentropy()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]


def train_and_evaluate(model_name, model):

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    training_time = time.time() - start_time

    test_loss, test_accuracy = model.evaluate(
        test_ds,
        verbose=0
    )

    results[model_name] = {
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "parameters": model.count_params(),
        "training_time_seconds": training_time,
        "history": history.history
    }

    trained_models[model_name] = model

    print("\nCompleted:", model_name)
    print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
    print("Training Time:", round(training_time / 60, 2), "minutes")

    return model


print("READY FOR MODEL TRAINING!")

In [ ]:
print("Train dataset:", train_ds)
print("Number of classes:", num_classes)
print("Class names:", class_names)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import gc

tf.keras.backend.clear_session()
gc.collect()

# Load pretrained ResNet50
base_model = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers for faster training
base_model.trainable = False

# Build ResNet50 classifier
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = tf.keras.applications.resnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

resnet50_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="ResNet50"
)

# Compile
resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=loss_function,
    metrics=["accuracy"]
)

print("ResNet50 model created successfully!")
print("Total parameters:", f"{resnet50_model.count_params():,}")

In [ ]:
resnet50_model = train_and_evaluate(
    "ResNet50",
    resnet50_model
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import gc

tf.keras.backend.clear_session()
gc.collect()

# Load pretrained DenseNet121
base_model = tf.keras.applications.DenseNet121(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# Build model
inputs = tf.keras.Input(shape=(224, 224, 3))

# Data augmentation
x = data_augmentation(inputs)

# DenseNet preprocessing
x = tf.keras.applications.densenet.preprocess_input(x)

# Pretrained DenseNet
x = base_model(x, training=False)

# Classification head
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

densenet121_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="DenseNet121"
)

# Compile model
densenet121_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=loss_function,
    metrics=["accuracy"]
)

print("DenseNet121 model created successfully!")
print("Total parameters:", f"{densenet121_model.count_params():,}")

In [ ]:
densenet121_model = train_and_evaluate(
    "DenseNet121",
    densenet121_model
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import gc

tf.keras.backend.clear_session()
gc.collect()

# Load pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# Build model
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

mobilenet_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2"
)

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=loss_function,
    metrics=["accuracy"]
)

print("MobileNetV2 ready!")
print("Total parameters:", f"{mobilenet_model.count_params():,}")

In [ ]:
import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "ResNet50",
        "DenseNet121",
        "MobileNetV2",
        "EfficientNetB0",
        "ConvNeXtTiny",
        "CBAM Attention CNN",
        "Vision Transformer"
    ],

    "Architecture Type": [
        "Conventional CNN",
        "Dense CNN",
        "Lightweight CNN",
        "Efficient CNN",
        "Modern CNN",
        "Attention-based CNN",
        "Vision Transformer"
    ],

    "Test Accuracy (%)": [
        88.69,
        75.12,
        91.50,
        80.54,
        63.69,
        23.67,
        32.80
    ],

    "Training Time (minutes)": [
        3.61,
        2.83,
        1.78,
        2.21,
        6.30,
        2.36,
        1.18
    ]
})

final_results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.bar(
    final_results["Model"],
    final_results["Test Accuracy (%)"]
)

plt.title("Test Accuracy Comparison of Deep Learning Models")
plt.xlabel("Model")
plt.ylabel("Test Accuracy (%)")

plt.xticks(rotation=30)

plt.ylim(0, 100)

plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import gc

tf.keras.backend.clear_session()
gc.collect()

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

best_model = tf.keras.Model(inputs, outputs)

best_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=loss_function,
    metrics=["accuracy"]
)

print("Best model ready!")

In [ ]:
history_best = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    verbose=1
)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy * 100, "%")

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Get true labels
y_true = []

for images, labels in test_ds:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

# Predictions
y_prob = best_model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

# Classification report
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

fig, ax = plt.subplots(figsize=(10, 8))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

display.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title("Confusion Matrix - MobileNetV2")
plt.show()

In [ ]:
print("MobileNetV2 Parameters:", f"{best_model.count_params():,}")

In [ ]:
print("MobileNetV2 Parameters:", f"{best_model.count_params():,}")

In [ ]:
final_comparison = pd.DataFrame({
    "Model": [
        "ResNet50",
        "DenseNet121",
        "MobileNetV2",
        "EfficientNetB0",
        "ConvNeXtTiny",
        "CBAM Attention CNN",
        "Vision Transformer"
    ],

    "Architecture Type": [
        "CNN",
        "Dense CNN",
        "Lightweight CNN",
        "Efficient CNN",
        "Modern CNN",
        "Attention-based CNN",
        "Vision Transformer"
    ],

    "Test Accuracy (%)": [
        88.69,
        75.12,
        91.50,
        80.54,
        63.69,
        23.67,
        32.80
    ],

    "Training Time (minutes)": [
        3.61,
        2.83,
        1.78,
        2.21,
        6.30,
        2.36,
        1.18
    ]
})

final_comparison = final_comparison.sort_values(
    "Test Accuracy (%)",
    ascending=False
)

final_comparison

In [ ]:
plt.figure(figsize=(10, 7))

plt.scatter(
    final_comparison["Training Time (minutes)"],
    final_comparison["Test Accuracy (%)"],
    s=120
)

for i, model in enumerate(final_comparison["Model"]):
    plt.annotate(
        model,
        (
            final_comparison["Training Time (minutes)"].iloc[i],
            final_comparison["Test Accuracy (%)"].iloc[i]
        ),
        fontsize=9
    )

plt.xlabel("Training Time (Minutes)")
plt.ylabel("Test Accuracy (%)")
plt.title("Accuracy vs Training Time Comparison")

plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
ranking = final_comparison[
    ["Model", "Test Accuracy (%)", "Training Time (minutes)"]
].sort_values(
    "Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

ranking.index = ranking.index + 1

print("MODEL PERFORMANCE RANKING")
ranking

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd
import gc

# Based on previous successful data preparation (cell -1P_rcBmWh69),
# num_classes is 8.
num_classes = 8

# Results list
parameter_results = []


# ============================================================
# 1. RESNET50
# ============================================================

base = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)

base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

parameter_results.append({
    "Model": "ResNet50",
    "Parameters": model.count_params()
})

del model, base
gc.collect()


# ============================================================
# 2. DENSENET121
# ============================================================

base = tf.keras.applications.DenseNet121(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

parameter_results.append({
    "Model": "DenseNet121",
    "Parameters": model.count_params()
})

del model, base
gc.collect()


# ============================================================
# 3. MOBILENETV2
# ============================================================

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

parameter_results.append({
    "Model": "MobileNetV2",
    "Parameters": model.count_params()
})

del model, base
gc.collect()


# ============================================================
# 4. EFFICIENTNETB0
# ============================================================

base = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

parameter_results.append({
    "Model": "EfficientNetB0",
    "Parameters": model.count_params()
})

del model, base
gc.collect()


# ============================================================
# 5. CONVNEXT TINY
# ============================================================

base = tf.keras.applications.ConvNeXtTiny(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

parameter_results.append({
    "Model": "ConvNeXtTiny",
    "Parameters": model.count_params()
})

del model, base
gc.collect()


# ============================================================
# SHOW PRETRAINED MODEL RESULTS
# ============================================================

parameter_df = pd.DataFrame(parameter_results)

parameter_df["Parameters (Millions)"] = (
    parameter_df["Parameters"] / 1_000_000
).round(2)

parameter_df

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd
import gc

IMG_SIZE = 224
num_classes = 8


# ============================================================
# CBAM BLOCK
# ============================================================

def cbam_block(inputs, reduction_ratio=8):

    channels = int(inputs.shape[-1])

    # Channel attention
    avg_pool = layers.GlobalAveragePooling2D()(inputs)
    max_pool = layers.GlobalMaxPooling2D()(inputs)

    shared_dense_1 = layers.Dense(
        max(channels // reduction_ratio, 1),
        activation="relu"
    )

    shared_dense_2 = layers.Dense(channels)

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Add()([avg_out, max_out])
    channel_attention = layers.Activation("sigmoid")(channel_attention)
    channel_attention = layers.Reshape((1, 1, channels))(channel_attention)

    x = layers.Multiply()([inputs, channel_attention])

    # Spatial attention
    avg_spatial = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=-1, keepdims=True)
    )(x)

    max_spatial = layers.Lambda(
        lambda z: tf.reduce_max(z, axis=-1, keepdims=True)
    )(x)

    spatial = layers.Concatenate(axis=-1)(
        [avg_spatial, max_spatial]
    )

    spatial_attention = layers.Conv2D(
        1,
        kernel_size=7,
        padding="same",
        activation="sigmoid"
    )(spatial)

    return layers.Multiply()([x, spatial_attention])


# ============================================================
# CBAM ATTENTION CNN
# ============================================================

inputs = tf.keras.Input(shape=(224, 224, 3))

x = layers.Rescaling(1.0 / 255)(inputs)

x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = cbam_block(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = cbam_block(x)
x = layers.MaxPooling2D()(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

cbam_param_model = tf.keras.Model(
    inputs,
    outputs,
    name="CBAM_Attention_CNN"
)

cbam_params = cbam_param_model.count_params()

print("CBAM Attention CNN Parameters:", f"{cbam_params:,}")


# ============================================================
# CLEAN MEMORY BEFORE ViT
# ============================================================

del cbam_param_model
gc.collect()


# ============================================================
# PATCH LAYER
# ============================================================

class Patches(layers.Layer):

    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        patch_dims = patches.shape[-1]

        return tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )


# ============================================================
# PATCH ENCODER
# ============================================================

class PatchEncoder(layers.Layer):

    def __init__(
        self,
        num_patches,
        projection_dim,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_patches = num_patches

        self.projection = layers.Dense(
            projection_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patches):

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        return (
            self.projection(patches)
            + self.position_embedding(positions)
        )


# ============================================================
# VISION TRANSFORMER
# ============================================================

PATCH_SIZE = 16
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2

PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4


inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = layers.Rescaling(1.0 / 255)(inputs)

patches = Patches(PATCH_SIZE)(x)

encoded_patches = PatchEncoder(
    NUM_PATCHES,
    PROJECTION_DIM
)(patches)


for _ in range(TRANSFORMER_LAYERS):

    x1 = layers.LayerNormalization(
        epsilon=1e-6
    )(encoded_patches)

    attention_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=PROJECTION_DIM,
        dropout=0.1
    )(x1, x1)

    x2 = layers.Add()([
        attention_output,
        encoded_patches
    ])

    x3 = layers.LayerNormalization(
        epsilon=1e-6
    )(x2)

    x3 = layers.Dense(
        128,
        activation="gelu"
    )(x3)

    x3 = layers.Dropout(0.1)(x3)

    x3 = layers.Dense(
        64,
        activation="gelu"
    )(x3)

    encoded_patches = layers.Add()([
        x3,
        x2
    ])


representation = layers.LayerNormalization(
    epsilon=1e-6
)(encoded_patches)

representation = layers.GlobalAveragePooling1D()(
    representation
)

representation = layers.Dropout(0.3)(
    representation
)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(representation)

vit_param_model = tf.keras.Model(
    inputs,
    outputs,
    name="VisionTransformer"
)

vit_params = vit_param_model.count_params()

print("Vision Transformer Parameters:", f"{vit_params:,}")
custom_parameter_df = pd.DataFrame({
    "Model": [
        "CBAM Attention CNN",
        "Vision Transformer"
    ],
    "Parameters": [
        cbam_params,
        vit_params
    ]
})

custom_parameter_df["Parameters (Millions)"] = (
    custom_parameter_df["Parameters"] / 1_000_000
).round(2)

print("\nCustom Model Parameter Counts:")
display(custom_parameter_df)

In [ ]:

all_parameter_df = pd.concat(
    [parameter_df, custom_parameter_df],
    ignore_index=True
)

all_parameter_df = all_parameter_df.sort_values(
    "Parameters",
    ascending=True
).reset_index(drop=True)

all_parameter_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    all_parameter_df["Model"],
    all_parameter_df["Parameters (Millions)"]
)

plt.xlabel("Model")
plt.ylabel("Parameters (Millions)")
plt.title("Parameter Count Comparison of Deep Learning Models")

plt.xticks(rotation=45)

for i, value in enumerate(all_parameter_df["Parameters (Millions)"]):
    plt.text(
        i,
        value,
        str(value),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
print("Available important variables:")

for var in [
    "best_model",
    "resnet50_model",
    "densenet121_model",
    "mobilenetv2_model",
    "efficientnetb0_model",
    "convnexttiny_model"
]:
    print(var, ":", var in globals())

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import time
import gc
import pandas as pd

IMG_SIZE = 224
num_classes = 8

# One sample batch for inference timing
dummy_input = tf.random.uniform((1, IMG_SIZE, IMG_SIZE, 3))

def create_classifier(base_model):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

In [ ]:
# ============================================================
# CREATE MODELS AND MEASURE INFERENCE TIME
# ============================================================

inference_results = []

model_builders = {
    "ResNet50": lambda: tf.keras.applications.ResNet50(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None
    ),

    "DenseNet121": lambda: tf.keras.applications.DenseNet121(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None
    ),

    "MobileNetV2": lambda: tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None
    ),

    "EfficientNetB0": lambda: tf.keras.applications.EfficientNetB0(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None
    ),

    "ConvNeXtTiny": lambda: tf.keras.applications.ConvNeXtTiny(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None
    )
}


for name, builder in model_builders.items():

    print(f"Measuring inference time: {name}")

    base = builder()
    model = create_classifier(base)

    # Warm-up runs
    for _ in range(5):
        _ = model(dummy_input, training=False)

    # Measure 20 inference runs
    start_time = time.perf_counter()

    for _ in range(20):
        _ = model(dummy_input, training=False)

    end_time = time.perf_counter()

    average_time_ms = (
        (end_time - start_time) / 20
    ) * 1000

    inference_results.append({
        "Model": name,
        "Average Inference Time (ms)": round(
            average_time_ms, 2
        )
    })

    print(
        f"{name}: {average_time_ms:.2f} ms"
    )

    del model
    del base
    gc.collect()


inference_df = pd.DataFrame(
    inference_results
).sort_values(
    "Average Inference Time (ms)"
).reset_index(drop=True)

print("\nInference Time Comparison:")
display(inference_df)

In [ ]:
# ============================================================
# CBAM + VISION TRANSFORMER INFERENCE TIME
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers
import time
import gc
import pandas as pd

IMG_SIZE = 224
num_classes = 8

dummy_input = tf.random.uniform((1, IMG_SIZE, IMG_SIZE, 3))


# ============================================================
# CBAM BLOCK
# ============================================================

def cbam_block(inputs, reduction_ratio=8):

    channels = int(inputs.shape[-1])

    # Channel Attention
    avg_pool = layers.GlobalAveragePooling2D()(inputs)
    max_pool = layers.GlobalMaxPooling2D()(inputs)

    shared_dense_1 = layers.Dense(
        max(channels // reduction_ratio, 1),
        activation="relu"
    )

    shared_dense_2 = layers.Dense(channels)

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Add()([avg_out, max_out])
    channel_attention = layers.Activation("sigmoid")(channel_attention)
    channel_attention = layers.Reshape((1, 1, channels))(channel_attention)

    x = layers.Multiply()([inputs, channel_attention])

    # Spatial Attention
    avg_spatial = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=-1, keepdims=True)
    )(x)

    max_spatial = layers.Lambda(
        lambda z: tf.reduce_max(z, axis=-1, keepdims=True)
    )(x)

    spatial = layers.Concatenate(axis=-1)(
        [avg_spatial, max_spatial]
    )

    spatial_attention = layers.Conv2D(
        1,
        kernel_size=7,
        padding="same",
        activation="sigmoid"
    )(spatial)

    return layers.Multiply()([x, spatial_attention])


# ============================================================
# CREATE CBAM ATTENTION CNN
# ============================================================

def create_cbam_model():

    inputs = tf.keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    x = layers.Rescaling(1.0 / 255)(inputs)

    x = layers.Conv2D(
        32, 3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(
        64, 3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)
    x = cbam_block(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(
        128, 3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)
    x = cbam_block(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    return tf.keras.Model(
        inputs,
        outputs,
        name="CBAM_Attention_CNN"
    )


# ============================================================
# PATCH + PATCH ENCODER FOR ViT
# ============================================================

class Patches(layers.Layer):

    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        patch_dims = patches.shape[-1]

        return tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )


class PatchEncoder(layers.Layer):

    def __init__(
        self,
        num_patches,
        projection_dim,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_patches = num_patches

        self.projection = layers.Dense(
            projection_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patches):

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        return (
            self.projection(patches)
            + self.position_embedding(positions)
        )


# ============================================================
# CREATE VISION TRANSFORMER
# ============================================================

def create_vit_model():

    PATCH_SIZE = 16
    NUM_PATCHES = (
        IMG_SIZE // PATCH_SIZE
    ) ** 2

    PROJECTION_DIM = 64
    NUM_HEADS = 4
    TRANSFORMER_LAYERS = 4

    inputs = tf.keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    x = layers.Rescaling(
        1.0 / 255
    )(inputs)

    patches = Patches(
        PATCH_SIZE
    )(x)

    encoded_patches = PatchEncoder(
        NUM_PATCHES,
        PROJECTION_DIM
    )(patches)

    for _ in range(
        TRANSFORMER_LAYERS
    ):

        x1 = layers.LayerNormalization(
            epsilon=1e-6
        )(encoded_patches)

        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=PROJECTION_DIM,
            dropout=0.1
        )(x1, x1)

        x2 = layers.Add()([
            attention_output,
            encoded_patches
        ])

        x3 = layers.LayerNormalization(
            epsilon=1e-6
        )(x2)

        x3 = layers.Dense(
            128,
            activation="gelu"
        )(x3)

        x3 = layers.Dropout(
            0.1
        )(x3)

        x3 = layers.Dense(
            64,
            activation="gelu"
        )(x3)

        encoded_patches = layers.Add()([
            x3,
            x2
        ])

    representation = layers.LayerNormalization(
        epsilon=1e-6
    )(encoded_patches)

    representation = layers.GlobalAveragePooling1D()(
        representation
    )

    representation = layers.Dropout(
        0.3
    )(representation)

    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(representation)

    return tf.keras.Model(
        inputs,
        outputs,
        name="VisionTransformer"
    )


# ============================================================
# INFERENCE TIME FUNCTION
# ============================================================

def measure_inference(model, name):

    # Warm-up
    for _ in range(5):
        _ = model(dummy_input, training=False)

    # Timing
    start = time.perf_counter()

    for _ in range(20):
        _ = model(dummy_input, training=False)

    end = time.perf_counter()

    avg_time_ms = (
        (end - start) / 20
    ) * 1000

    print(
        f"{name}: {avg_time_ms:.2f} ms"
    )

    return avg_time_ms


# ============================================================
# CBAM
# ============================================================

print("Measuring CBAM Attention CNN...")

cbam_inference_model = create_cbam_model()

cbam_time = measure_inference(
    cbam_inference_model,
    "CBAM Attention CNN"
)

del cbam_inference_model
gc.collect()


# ============================================================
# VISION TRANSFORMER
# ============================================================

print("\nMeasuring Vision Transformer...")

vit_inference_model = create_vit_model()

vit_time = measure_inference(
    vit_inference_model,
    "Vision Transformer"
)

del vit_inference_model
gc.collect()


# ============================================================
# RESULTS
# ============================================================

custom_inference_df = pd.DataFrame({
    "Model": [
        "CBAM Attention CNN",
        "Vision Transformer"
    ],

    "Average Inference Time (ms)": [
        round(cbam_time, 2),
        round(vit_time, 2)
    ]
})

display(custom_inference_df)

In [ ]:
all_inference_df = pd.concat(
    [inference_df, custom_inference_df],
    ignore_index=True
)

all_inference_df = all_inference_df.sort_values(
    "Average Inference Time (ms)"
).reset_index(drop=True)

display(all_inference_df)

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    all_inference_df["Model"],
    all_inference_df["Average Inference Time (ms)"]
)

plt.xlabel("Model")
plt.ylabel("Average Inference Time (ms)")
plt.title("Inference Time Comparison of Deep Learning Models")

plt.xticks(rotation=45)

for i, value in enumerate(
    all_inference_df["Average Inference Time (ms)"]
):
    plt.text(
        i,
        value,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

import os
import glob
import zipfile
from pathlib import Path
import pandas as pd

# Re-extract dataset to ensure data is present
zip_files = glob.glob("/content/*.zip")
if zip_files:
    zip_path = zip_files[0]
    extract_path = Path("/kaggle/input/datasets/paultimothymooney/blood-cells")

    # Ensure the directory is clean before re-extracting
    if extract_path.exists():
        import shutil
        shutil.rmtree(extract_path)
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Recreate df_unique from the extracted files
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_files = [
        f for f in extract_path.rglob("*")
        if f.is_file() and f.suffix.lower() in image_extensions
    ]

    df_unique = pd.DataFrame({
        "filepath": [str(f) for f in image_files],
        "parent_folder": [f.parent.name for f in image_files]
    })
else:
    print("No zip file found for extraction. df_unique will be empty.")
    df_unique = pd.DataFrame(columns=["filepath", "parent_folder"])

# Settings
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

# Get class names
class_names = sorted(df_unique["parent_folder"].unique()) if not df_unique.empty else []
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

# Convert class names to numerical labels
class_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

df = df_unique.copy()

if not df.empty:
    df["label"] = df["parent_folder"].map(class_to_index)

    # 70% Train, 30% temporary
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df["label"],
        random_state=SEED
    )

    # Split remaining 30% into 15% validation and 15% test
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=SEED
    )

    print("\nDataset Split:")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))
else:
    print("\nDataFrame df is empty, skipping dataset split.")
    train_df = pd.DataFrame()
    val_df = pd.DataFrame()
    test_df = pd.DataFrame()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


def create_dataset(dataframe, shuffle=False):

    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            len(dataframe),
            seed=SEED
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(
    train_df,
    shuffle=True
)

val_ds = create_dataset(val_df)

test_ds = create_dataset(test_df)

print("Datasets recreated successfully.")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# ============================================================
# MOBILENETV2 MODEL
# ============================================================

IMG_SIZE = 224
num_classes = 8

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False


# Build model
inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

mobilenet_final = tf.keras.Model(
    inputs,
    outputs,
    name="MobileNetV2_Final"
)


# Compile
mobilenet_final.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


mobilenet_final.summary()

In [ ]:
import time

print("=" * 60)
print("Training MobileNetV2 for final evaluation")
print("=" * 60)

start_time = time.time()

history_mobilenet = mobilenet_final.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

training_time = (
    time.time() - start_time
) / 60

print("\nTraining completed")
print(
    f"Training Time: {training_time:.2f} minutes"
)

In [ ]:
print(mobilenet_final)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = 224
num_classes = 8

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Load pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# Build the final model
inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

mobilenet_final = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2_Final"
)

# Compile
mobilenet_final.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("MobileNetV2 model created successfully!")
print("Number of classes:", num_classes)

mobilenet_final.summary()

In [ ]:
import time

print("=" * 60)
print("Training MobileNetV2 for final evaluation")
print("=" * 60)

start_time = time.time()

history_mobilenet = mobilenet_final.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

training_time = (time.time() - start_time) / 60

print("\nTraining completed")
print(f"Training Time: {training_time:.2f} minutes")

In [ ]:
import os

DATASET_PATH = "/kaggle/input/datasets/paultimothymooney/blood-cells"

print("Dataset exists:", os.path.exists(DATASET_PATH))

if os.path.exists(DATASET_PATH):
    print("Folders:")
    print(os.listdir(DATASET_PATH)[:20])

In [ ]:
import os
import pandas as pd

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

data = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(image_extensions):

            filepath = os.path.join(root, file)

            parent_folder = os.path.basename(
                os.path.dirname(filepath)
            )

            data.append({
                "filepath": filepath,
                "filename": file,
                "parent_folder": parent_folder
            })

df_unique = pd.DataFrame(data)

print("Total images:", len(df_unique))
print()
print(df_unique["parent_folder"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

class_names = sorted(
    df_unique["parent_folder"].unique()
)

num_classes = len(class_names)

class_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

df = df_unique.copy()

df["label"] = df["parent_folder"].map(
    class_to_index
)

# 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

# 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print()
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


def create_dataset(dataframe, shuffle=False):

    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(
    train_df,
    shuffle=True
)

val_ds = create_dataset(
    val_df
)

test_ds = create_dataset(
    test_df
)

print("Train dataset:", train_ds)
print("Validation dataset:", val_ds)
print("Test dataset:", test_ds)

In [ ]:
print("mobilenet_final exists:", "mobilenet_final" in globals())
print("train_ds exists:", "train_ds" in globals())
print("val_ds exists:", "val_ds" in globals())
print("test_ds exists:", "test_ds" in globals())

In [ ]:
import time

print("=" * 60)
print("Training MobileNetV2 for final evaluation")
print("=" * 60)

start_time = time.time()

history_mobilenet = mobilenet_final.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

training_time = (time.time() - start_time) / 60

print("\nTraining completed successfully!")
print(f"Training Time: {training_time:.2f} minutes")

In [ ]:
import os

print(
    "Model file exists:",
    os.path.exists("/content/mobilenet_final.keras")
)

In [ ]:
test_loss, test_accuracy = mobilenet_final.evaluate(
    test_ds,
    verbose=1
)

print("\nFinal Test Results")
print("=" * 40)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
# ============================================================
# GET TRUE LABELS AND PREDICTION PROBABILITIES
# ============================================================

import numpy as np
from sklearn.preprocessing import label_binarize

y_true = []
y_pred_proba = []

for images, labels in test_ds:

    # Get prediction probabilities
    predictions = mobilenet_final.predict(
        images,
        verbose=0
    )

    y_pred_proba.extend(predictions)
    y_true.extend(labels.numpy())

# Convert to NumPy arrays
y_true = np.array(y_true)
y_pred_proba = np.array(y_pred_proba)

print("True labels shape:", y_true.shape)
print("Prediction probabilities shape:", y_pred_proba.shape)

print("\nExample true label:", y_true[0])
print("Example prediction probabilities:")
print(y_pred_proba[0])

In [ ]:
print(y_pred_proba.shape)
print(len(class_names))
print(class_names)

In [ ]:
# ============================================================
# MULTICLASS ROC-AUC ANALYSIS
# ============================================================

from sklearn.metrics import (
    roc_curve,
    auc,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Convert true labels to one-hot/binary format
y_true_bin = label_binarize(
    y_true,
    classes=list(range(num_classes))
)
# ============================================================
# MULTICLASS ROC-AUC ANALYSIS
# ============================================================

from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Number of classes comes from the prediction probabilities
num_classes = y_pred_proba.shape[1]

print("Number of classes:", num_classes)
print("y_true shape:", np.shape(y_true))
print("y_pred_proba shape:", np.shape(y_pred_proba))

# ------------------------------------------------------------
# IMPORTANT:
# Get class names
# ------------------------------------------------------------

print("Class names:", class_names)

# Check that class_names also contains 8 classes
if len(class_names) != num_classes:
    raise ValueError(
        f"Mismatch: y_pred_proba has {num_classes} classes, "
        f"but class_names has {len(class_names)} classes."
    )

# ------------------------------------------------------------
# Convert true labels to binary format
# ------------------------------------------------------------

y_true_bin = label_binarize(
    y_true,
    classes=list(range(num_classes))
)

print("y_true_bin shape:", y_true_bin.shape)

# ------------------------------------------------------------
# Overall multiclass ROC-AUC
# ------------------------------------------------------------

macro_auc = roc_auc_score(
    y_true_bin,
    y_pred_proba,
    average="macro",
    multi_class="ovr"
)

weighted_auc = roc_auc_score(
    y_true_bin,
    y_pred_proba,
    average="weighted",
    multi_class="ovr"
)

print("=" * 50)
print("MULTICLASS ROC-AUC RESULTS")
print("=" * 50)

print(f"Macro ROC-AUC: {macro_auc:.4f}")
print(f"Weighted ROC-AUC: {weighted_auc:.4f}")


# ============================================================
# ROC CURVE FOR EACH BLOOD CELL CLASS
# ============================================================

plt.figure(figsize=(10, 8))

for i in range(num_classes):

    fpr, tpr, _ = roc_curve(
        y_true_bin[:, i],
        y_pred_proba[:, i]
    )

    class_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{class_names[i]} (AUC = {class_auc:.3f})"
    )


# Random classifier reference line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curves for MobileNetV2 Blood Cell Classification"
)

plt.legend(loc="lower right")

plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
sample_images, sample_labels = next(iter(test_ds))

sample_image = sample_images[0:1]
sample_label = sample_labels[0].numpy()

print("True Class:", class_names[sample_label])

plt.figure(figsize=(5, 5))
plt.imshow(sample_images[0].numpy().astype("uint8"))
plt.title(f"True Class: {class_names[sample_label]}")
plt.axis("off")
plt.show()

In [ ]:
for i, layer in enumerate(mobilenet_final.layers):
    print(i, layer.name, type(layer).__name__)

In [ ]:
for layer in mobilenet_final.layers:
    if isinstance(layer, tf.keras.Model):
        print("Nested model:", layer.name)

        print("\nLast 15 layers:")
        for sublayer in layer.layers[-15:]:
            print(
                sublayer.name,
                "|",
                type(sublayer).__name__
            )

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
IMG_SIZE = 224
num_classes = 8

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False


# ============================================================
# CREATE FINAL CLASSIFICATION MODEL
# ============================================================

inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = data_augmentation(inputs)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs,
    outputs
)


# Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = data_augmentation(inputs)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
# ============================================================
# GRAD-CAM MODEL
# ============================================================

last_conv_layer = base_model.get_layer("out_relu")

grad_model = tf.keras.models.Model(
    inputs=base_model.input,
    outputs=[
        last_conv_layer.output,
        base_model.output
    ]
)

print("Grad-CAM layer:", last_conv_layer.name)

In [ ]:
# ============================================================
# GENERATE GRAD-CAM
# ============================================================

# Get MobileNetV2 feature maps
with tf.GradientTape() as tape:

    conv_outputs = grad_model(sample_image)

    feature_maps = conv_outputs[0]

    # Send feature maps through the classification head
    x = feature_maps
    x = model.layers[-3](x)   # GlobalAveragePooling2D
    x = model.layers[-2](x)   # Dropout
    predictions = model.layers[-1](x)

    predicted_class = tf.argmax(
        predictions[0]
    )

    class_score = predictions[:, predicted_class]


# Calculate gradients
grads = tape.gradient(
    class_score,
    feature_maps
)

# Average gradients
pooled_grads = tf.reduce_mean(
    grads,
    axis=(0, 1, 2)
)

# Remove batch dimension
feature_maps = feature_maps[0]

# Weighted combination
heatmap = feature_maps @ pooled_grads[..., tf.newaxis]

heatmap = tf.squeeze(heatmap)

# ReLU
heatmap = tf.maximum(
    heatmap,
    0
)

# Normalize
heatmap = heatmap / (
    tf.reduce_max(heatmap) + 1e-8
)

heatmap = heatmap.numpy()

print("Heatmap generated successfully!")
print("Predicted class:", predicted_class.numpy())

In [ ]:
# ============================================================
# VISUALIZE GRAD-CAM
# ============================================================

original_image = sample_image[0].numpy()

# Convert image for display
display_image = np.clip(
    original_image,
    0,
    255
).astype("uint8")

# Resize heatmap to image size
heatmap_resized = cv2.resize(
    heatmap,
    (IMG_SIZE, IMG_SIZE)
)

# Convert heatmap to colored map
heatmap_uint8 = np.uint8(
    255 * heatmap_resized
)

heatmap_color = cv2.applyColorMap(
    heatmap_uint8,
    cv2.COLORMAP_JET
)

# OpenCV uses BGR, convert to RGB
heatmap_color = cv2.cvtColor(
    heatmap_color,
    cv2.COLOR_BGR2RGB
)

# Create overlay
superimposed_image = (
    0.6 * display_image
    + 0.4 * heatmap_color
)

superimposed_image = np.clip(
    superimposed_image,
    0,
    255
).astype("uint8")


# ============================================================
# DISPLAY RESULTS
# ============================================================

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)

plt.imshow(display_image)

plt.title(
    f"Original\nTrue: {class_names[sample_label]}"
)

plt.axis("off")


plt.subplot(1, 3, 2)

plt.imshow(
    heatmap_resized,
    cmap="jet"
)

plt.title("Grad-CAM Heatmap")

plt.axis("off")


plt.subplot(1, 3, 3)

plt.imshow(superimposed_image)

plt.title(
    f"Grad-CAM Overlay\nPredicted: {class_names[predicted_class.numpy()]}"
)

plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
!pip install -q lime

from lime import lime_image
from skimage.segmentation import mark_boundaries

print("LIME installed successfully!")

In [ ]:
# ============================================================
# LIME EXPLAINABILITY FOR MOBILENETV2
# ============================================================

from lime import lime_image
from skimage.segmentation import mark_boundaries
import numpy as np
import matplotlib.pyplot as plt

# LIME prediction function
def predict_for_lime(images):

    images = np.array(images).astype("float32")

    # Dataset images are already in the same format used by the model
    predictions = mobilenet_final.predict(
        images,
        verbose=0
    )

    return predictions


# Convert sample image to NumPy
lime_image_input = sample_image[0].numpy()

print("LIME input shape:", lime_image_input.shape)

# Create LIME explainer
explainer = lime_image.LimeImageExplainer()

print("Generating LIME explanation...")

# Generate explanation
explanation = explainer.explain_instance(
    lime_image_input,
    predict_for_lime,
    top_labels=3,
    hide_color=0,
    num_samples=500
)

print("LIME explanation generated successfully!")

# Show predicted class
lime_prediction = predict_for_lime(
    np.expand_dims(lime_image_input, axis=0)
)

lime_predicted_class = np.argmax(lime_prediction[0])

print("True class:", class_names[sample_label])
print("Predicted class:", class_names[lime_predicted_class])
print(
    "Prediction confidence:",
    f"{lime_prediction[0][lime_predicted_class] * 100:.2f}%"
)

In [ ]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "ResNet50",
        "DenseNet121",
        "MobileNetV2",
        "EfficientNetB0",
        "ConvNeXtTiny",
        "CBAM Attention CNN",
        "Vision Transformer"
    ],

    "Test Accuracy (%)": [
        88.69,
        75.12,
        91.85,
        80.54,
        63.69,
        23.67,
        32.80
    ],

    "Training Time (minutes)": [
        3.61,
        2.83,
        3.07,
        2.21,
        6.30,
        2.36,
        1.18
    ],

    "Inference Time (ms)": [
        225.94,
        427.87,
        177.30,
        276.73,
        2529.41,
        133.57,
        161.88
    ]
})

# Sort by test accuracy
final_results = final_results.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

final_results

In [ ]:
# ============================================================
# VISUALIZE LIME EXPLANATION
# ============================================================

# Get explanation for predicted class
temp, mask = explanation.get_image_and_mask(
    lime_predicted_class,
    positive_only=True,
    num_features=10,
    hide_rest=False
)

# Display original image and LIME explanation
plt.figure(figsize=(12, 5))

# Original image
plt.subplot(1, 2, 1)

plt.imshow(lime_image_input.astype("uint8"))

plt.title(
    f"Original Image\nTrue: {class_names[sample_label]}"
)

plt.axis("off")


# LIME explanation
plt.subplot(1, 2, 2)

plt.imshow(
    mark_boundaries(
        temp.astype("uint8"),
        mask
    )
)

plt.title(
    f"LIME Explanation\nPredicted: {class_names[lime_predicted_class]}"
)

plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "ResNet50",
        "DenseNet121",
        "MobileNetV2",
        "EfficientNetB0",
        "ConvNeXtTiny",
        "CBAM Attention CNN",
        "Vision Transformer"
    ],

    "Test Accuracy (%)": [
        88.69,
        75.12,
        91.85,
        80.54,
        63.69,
        23.67,
        32.80
    ],

    "Training Time (minutes)": [
        3.61,
        2.83,
        3.07,
        2.21,
        6.30,
        2.36,
        1.18
    ],

    "Inference Time (ms)": [
        225.94,
        427.87,
        177.30,
        276.73,
        2529.41,
        133.57,
        161.88
    ]
})

# Sort by test accuracy
final_results = final_results.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

final_results